## Import packages

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import anndata
from pym3c.hic import mcds2frac
from scipy.stats import pearsonr
import cooler
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pylab as plt
import matplotlib as mpl
import seaborn as sns
mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 80
mpl.rcParams['savefig.dpi']=300
from PyComplexHeatmap import *

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/HBA_pseudobulk"
indir=None
res=25000
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv"
pseudobulk_mcds_path = "/home/x-wding2/project_wubin/BG/pseudobulk/Leaf/Study+Name+donor+Region+Subclass.5kb.mcds" #"5kb", used to calculate correlation between mC and compartment score
pseudobulk_atac_adata_path=None
prefix='Subclass+Region' # used to define indir: {prefix}_compartment
group_col='Subclass' # Region # used to define figdir, and cell_types (group_order), match the col in metadata.columns
parent_col="Class" #used to sort the group_col (MajorType)
sort_cols=None
stratify_col = None #Subclass
stratify_figdir=None
palette_path = "/home/x-wding2/Projects/BG/metadata/HMBA_color_palette.xlsx"
split_order = ["Nonneuron", "Glut", "Splatter", "Subpallium GABA"]
var_dim5kb="5kb"
extract_group=None
query_group=None
normalize_per_cell=True
figdir=None
cpu = 16
gene_annotation=None
species='human'
kernel_name = "m3c"
excluded_groups=['Low quality cells','doublet','Unk']

extract_group = [4]
query_group = {"0": "HBA"}

extract_group = [4,3]
group_col = "Region"
stratify_col = "Subclass"

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/domain"
outname = "2.Class.domain_correlation.ipynb"


In [ ]:
if not workdir is None:
    workdir=os.path.expanduser(workdir)
if not indir is None:
    indir=os.path.expanduser(indir)
else:
    assert not workdir is None
    indir=os.path.join(workdir,f"{prefix}_domain")
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"domain",group_col)
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)

print(figdir,group_col)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
keep_chroms=[chrom for chrom in chrom_sizes.index if chrom not in ['chrX','chrY']]
chrom_sizes = chrom_sizes[keep_chroms]
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
# Filter out low quality cells
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
metadata.head()

In [ ]:
# chrom_sizes.index.values

In [ ]:
boundary=anndata.read_h5ad(os.path.join(figdir, "cell_boundary.h5ad"))
bound_count_ct = pd.read_csv(os.path.join(figdir, 'bound_count.tsv.gz'),sep='\t',index_col=0)
group_name=group_col if stratify_col is None else 'GroupID'
cell_count_ct = boundary.obs[group_name].value_counts().loc[bound_count_ct.columns.tolist()]
bound_prob_ct = bound_count_ct / cell_count_ct
del boundary
bound_prob_ct

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not sort_cols is None:
    metadata.sort_values(sort_cols,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)
sorted_groups=metadata[group_col].unique().tolist()
stratify2group={}
group_order=[]
if stratify_col is None:
    for g in sorted_groups:
        sname=g.replace(' ','_')
        if os.path.exists(os.path.join(indir,sname,'25K_domain.boundary.h5ad')):
            group_order.append(g)
else:
    metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str)
    stratify_values=metadata[stratify_col].unique().tolist()
    for s in stratify_values:
        s_name=s.replace(' ','_')
        for g in sorted_groups:
            g_name=g.replace(' ','_')
            ct=f'{s_name}+{g_name}'
            if os.path.exists(os.path.join(indir,ct,'25K_domain.boundary.h5ad')):
                if s not in stratify2group:
                    stratify2group[s]=[]
                stratify2group[s].append(g) # For example: keys are Subclass, values are Subclass+Region
                group_order.append(f"{s}+{g}")
group2stratify={}
for k in stratify2group:
    for g in stratify2group[k]:
        group2stratify[g]=k
print(len(group_order),group_order)
print(stratify2group)

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
group_colors=D[group_col].Hex.to_dict()
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    parent_colors['merged']='black'
    if stratify_col is None:
        group2parent=metadata.loc[:,[group_col,parent_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        D1=D[stratify_col].Hex.to_dict()
        stratify_colors={k: D1[k] for k in D1 if k in stratify_values}
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    group2parent['merged']='merged'
else:
    parent_colors=None
    group2parent={}
if not stratify_col is None:
    D1=D[stratify_col].Hex.to_dict()
    stratify_colors={k: D1[k] for k in D1 if k in stratify2group}
else:
    stratify_colors={}
print(group_colors)
print(parent_colors)
print(stratify_colors)
print(group2parent)

## Load data

In [ ]:
frac_dict=mcds2frac(pseudobulk_mcds_path=pseudobulk_mcds_path,
					   var_dim=var_dim5kb,res=res,mc_types=['CGN','CHN'],
					   extract_group=extract_group,query_group=query_group,filter_chroms=True,
					   normalize_per_cell=normalize_per_cell,boundary=False)
mcg=frac_dict['CGN']
mch=frac_dict['CHN']

In [ ]:
from pym3c.adata import merge_adata_regions
if not pseudobulk_atac_adata_path is None:
    atac_data=merge_adata_regions(pseudobulk_atac_adata_path,bin_size=5000,use_raw=True,
    		res=res,filter_chroms=True,boundary=False)
    atac_data=atac_data.T
    print(atac_data.shape)
else:
    atac_data=None

In [ ]:
df_bin=pd.read_csv(os.path.join(figdir,'bin_stats.tsv.gz'),sep='\t',index_col=0)
df_bin

In [ ]:
# from ALLCools.mcds import MCDS
# from ALLCools.mcds.utilities import calculate_posterior_mc_frac
# boundcg = pd.DataFrame(index=df_bin.index, columns=group_order)
# var_dim="5kb"
# mcds = MCDS.open(os.path.expanduser(pseudobulk_mcds_path), var_dim=var_dim)
# mcds['chrom10k_even'] = mcds[f'{var_dim}_chrom'].to_pandas().astype(str) + '_' + (mcds[f'{var_dim}_start'] // 10000 * 2).to_pandas().astype(str)
# mcds['chrom10k_odd'] = mcds[f'{var_dim}_chrom'].to_pandas().astype(str) + '_' + ((mcds[f'{var_dim}_start'] + 5000) // 10000 * 2 - 1).to_pandas().astype(str)
# mc = mcds[f'{var_dim}_da'].sel(count_type='mc', mc_type='CGN').to_pandas().T
# mc['chrom10k'] = mcds['chrom10k_odd'].to_pandas()
# col_order=mc['chrom10k'].unique().tolist()
# mc = mc.groupby('chrom10k').sum().T.loc[:,col_order]
# cov = mcds[f'{var_dim}_da'].sel(count_type='cov', mc_type='CGN').to_pandas().T
# cov['chrom10k'] = mcds['chrom10k_odd'].to_pandas()
# col_order=cov['chrom10k'].unique().tolist()
# cov = cov.groupby('chrom10k').sum().T.loc[:,col_order]
# binfilter = ['_'.join(xx.split('_')[:-1]) for xx in mc.columns]
# binfilter = [(len(xx)<6) and (xx not in ['chrM','chrX','chrY']) for xx in binfilter]
# print(np.sum(binfilter))
# mc.index = [col.replace('_', ' ') for col in mc.index.tolist()]
# mc = mc.loc[group_order, binfilter]
# cov.index = [col.replace('_', ' ') for col in cov.index.tolist()]
# cov = cov.loc[group_order, binfilter]
# print(mc.shape, cov.shape)
# # ratio.head()
# idx1 = df_bin['chrom'].astype(str) + '_' + (df_bin['start'] // 5000 - 2).astype(str)
# idx2 = df_bin['chrom'].astype(str) + '_' + (df_bin['start'] // 5000).astype(str)
# selb = (df_bin['start']%10000!=0)#  & idx1.isin(ratio.columns)
# mc1=(mc.loc[:, idx1[selb]].values + mc.loc[:, idx2[selb]].values)
# cov1=(cov.loc[:, idx1[selb]].values + cov.loc[:, idx2[selb]].values)
# ratio = calculate_posterior_mc_frac(mc1, cov1)[0]
# boundcg.loc[selb] = ratio.T


# mc = mcds[f'{var_dim}_da'].sel(count_type='mc', mc_type='CGN').to_pandas().T
# mc['chrom10k'] = mcds['chrom10k_even'].to_pandas()
# col_order=mc['chrom10k'].unique().tolist()
# mc = mc.groupby('chrom10k').sum().T.loc[:,col_order]
# cov = mcds[f'{var_dim}_da'].sel(count_type='cov', mc_type='CGN').to_pandas().T
# cov['chrom10k'] = mcds['chrom10k_even'].to_pandas()
# col_order=cov['chrom10k'].unique().tolist()
# cov = cov.groupby('chrom10k').sum().T.loc[:,col_order]
# binfilter = ['_'.join(xx.split('_')[:-1]) for xx in mc.columns]
# binfilter = [(len(xx)<6) and (xx not in ['chrM','chrX','chrY']) for xx in binfilter]
# print(np.sum(binfilter))
# mc.index = [col.replace('_', ' ') for col in mc.index.tolist()]
# mc = mc.loc[group_order, binfilter]
# cov.index = [col.replace('_', ' ') for col in cov.index.tolist()]
# cov = cov.loc[group_order, binfilter]
# print(mc.shape, cov.shape)

# idx1 = df_bin['chrom'].astype(str) + '_' + (df_bin['start'] // 5000 - 2).astype(str)
# idx2 = df_bin['chrom'].astype(str) + '_' + (df_bin['start'] // 5000).astype(str)
# selb = (df_bin['start']%10000==0) & idx1.isin(mc.columns)
# mc1=(mc.loc[:, idx1[selb]].values + mc.loc[:, idx2[selb]].values)
# cov1=(cov.loc[:, idx1[selb]].values + cov.loc[:, idx2[selb]].values)
# ratio = calculate_posterior_mc_frac(mc1, cov1)[0]
# boundcg.loc[selb] = ratio.T

In [ ]:
use_cols=[ct for ct in group_order if ct in mcg.columns.tolist()]
mcg=mcg.loc[:,use_cols]
mch=mch.loc[:,use_cols]

In [ ]:
rows=df_bin.index.tolist()
cols=list(set(bound_prob_ct.columns.tolist()) & set(mcg.columns.tolist()))
df_bin['mCG_corr'] = [pearsonr(xx, yy)[0] for xx,yy in zip(bound_prob_ct.loc[rows,cols].values, mcg.loc[rows,cols].values)]
df_bin['mCH_corr'] = [pearsonr(xx, yy)[0] for xx,yy in zip(bound_prob_ct.loc[rows,cols].values, mch.loc[rows,cols].values)]
if not pseudobulk_atac_adata_path is None:
    keep_cols=list(set(cols) & set(atac_data.columns.tolist()))
    atac=atac_data.loc[:,keep_cols]
    df_bin['ATAC_corr'] = [pearsonr(xx, yy)[0] for xx,yy in zip(bound_prob_ct.loc[rows,keep_cols].values, atac.loc[rows,keep_cols].values)]
df_bin.to_csv(os.path.join(figdir,'bin_stats_with_coor.tsv.gz'),sep='\t')
df_bin

## Plot for merged cell types (all Major cell types together)

In [ ]:
# df_bin=pd.read_csv(os.path.join(figdir,'bin_stats_with_coor.tsv.gz'),sep='\t',index_col=0)
# df_bin

In [ ]:
sig_bins=(df_bin['chi2filter'] & df_bin['chi2_stat_peak'] & df_bin['bklfilter'] & df_bin['ins_peak'] 
          & (df_bin['probdiff']>0.05) &(df_bin.boundary_strength > 0)) #& (df_bin['insfc']>1.5)
print(sig_bins.sum())

In [ ]:
group_name=group_col if stratify_col is None else 'GroupID'
df_cols=bound_prob_ct.columns.to_frame(name=group_name)
if not stratify_col is None:
    if stratify_figdir is None:
        df_cols[stratify_col]=df_cols[group_name].apply(lambda x:x.split('+')[0])
        df_cols[group_col]=df_cols[group_name].apply(lambda x:x.split('+')[1])
    else:
        df_cols[stratify_col]=df_cols[group_name].map(group2stratify)
        df_cols[group_col]=df_cols[group_name]
if not parent_col is None:
    df_cols[parent_col]=df_cols[group_name].map(group2parent)
    df_cols.sort_values([parent_col,group_name],inplace=True)
else:
    if split_order is None:
        df_cols.sort_values(group_name,inplace=True)
    else:
        df_cols=df_cols.loc[split_order]
df_cols.head()

In [ ]:
df_bin

In [ ]:
df_bin['position']=df_bin.index.to_series().apply(lambda x:x.split('_')[0]+':'+str(int(x.split('_')[1])*res)+'-'+str((int(x.split('_')[1])+1)*res))
cols=bound_prob_ct.columns.tolist()
df_bin['Group']=df_bin.index.to_series().map(bound_prob_ct.apply(lambda x:cols[np.argmax(x)], axis=1).to_dict())

In [ ]:
df_rows=df_bin.loc[sig_bins]
if not stratify_col is None:
    if stratify_figdir is None:
        df_rows[stratify_col]=df_rows.Group.apply(lambda x:x.split('+')[0])
        df_rows[group_col]=df_rows.Group.apply(lambda x:x.split('+')[1])
    else:
        df_rows[stratify_col]=df_rows.Group.map(group2stratify)
        df_rows[group_col]=df_rows.Group
if not parent_col is None:
    df_rows[parent_col]=df_rows.Group.map(group2parent)
df_rows.head()

In [ ]:
if not parent_col is None and not stratify_col is None:
    parent2stratify=df_rows.loc[:,[parent_col,stratify_col]].drop_duplicates().groupby(parent_col)[stratify_col].apply(lambda x:x.unique().tolist()).to_dict()
    row_order=[]
    for parent in parent2stratify:
        for stratify in parent2stratify[parent]:
            for g in stratify2group[stratify]:
                if stratify_figdir is None:
                    rows=df_rows.loc[df_rows['Group']==f"{stratify}+{g}"].sort_values('probdiff',ascending=False).index.tolist()
                else:
                    rows=df_rows.loc[df_rows['Group']==g].sort_values('probdiff',ascending=False).index.tolist()
                row_order.extend(rows)
elif not stratify_col is None:
    row_order=[]
    for stratify in stratify2group:
        for g in stratify2group[stratify]:
            if stratify_figdir is None:
                rows=df_rows.loc[df_rows['Group']==f"{stratify}+{g}"].sort_values('probdiff',ascending=False).index.tolist()
            else:
                rows=df_rows.loc[df_rows['Group']==g].sort_values('probdiff',ascending=False).index.tolist()
            row_order.extend(rows)
else:
    row_order=[]
    for g in df_cols[group_name].tolist():
        rows=df_rows.loc[df_rows.Group==g].sort_values('probdiff',ascending=False).index.tolist()
        row_order.extend(rows)
df_rows=df_rows.loc[row_order]
df_rows

In [ ]:
df_cols=df_cols.loc[df_rows.Group.unique().tolist()]
df_cols
# row_order=[]
# col_order=[g for g in df_cols[group_name].tolist() if g in bound_prob_ct.columns.tolist()]
# for g in col_order:
#     rows=df_rows.loc[df_rows.Group==g].sort_values('probdiff',ascending=False).index.tolist()
#     row_order.extend(rows)
# df_rows=df_rows.loc[row_order]
# df_rows

In [ ]:
tmpcg = mcg.loc[df_rows.index,df_cols.index]
tmpch = mch.loc[df_rows.index,df_cols.index]
tmp3c = bound_prob_ct.loc[df_rows.index,df_cols.index]

In [ ]:
tmp3c.head()

In [ ]:
print(tmp3c.shape,tmpcg.shape,tmpch.shape)
if not atac_data is None:
    keep_cols=list(set(df_cols.index.tolist()) & set(atac_data.columns.tolist()))
    atac=atac_data.loc[df_rows.index,keep_cols]
    tmpcg = mcg.loc[df_rows.index,keep_cols]
    tmpch = mch.loc[df_rows.index,keep_cols]
    tmp3c = bound_prob_ct.loc[df_rows.index,keep_cols]
    df_cols=df_cols.loc[keep_cols]
    print(atac.shape)

In [ ]:
col_ha_dict={}
if not parent_col is None:
    tmp_colors={}
    for p in df_rows[parent_col].unique():
        tmp_colors[p]=parent_colors[p]
    row_ha_dict={
                parent_col:anno_simple(df_rows[parent_col],colors=tmp_colors,legend=False),
                }
    row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
             verbose=0,label_kws=dict(visible=False))
    row_split=df_rows[parent_col]
    row_split_order=df_cols[parent_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_cols[parent_col].tolist()]

    col_ha_dict[parent_col]=anno_simple(
                    df_cols[parent_col],#height=4,
                     colors=tmp_colors,
                     add_text=False,text_kws={'fontweight':'bold','color':'white'},legend=True)
    col_split=df_cols[parent_col]
    col_split_order=df_cols[parent_col].unique().tolist() if split_order is None else [o for o in split_order if o in df_cols[parent_col].tolist()]
else:
    row_ha=None
    row_split=None
    row_split_order=None
    col_split=None
if not stratify_col is None:
    tmp_colors={}
    for g in df_cols[group_col].unique():
        tmp_colors[g]=group_colors[g]
    col_ha_dict[group_col]=anno_simple(
                    df_cols[group_col],#height=5,
                     colors=tmp_colors,
                     add_text=False,text_kws={'fontweight':'bold','color':'white'},legend=True)
    show_colnames=True
else:
    show_colnames=True
if len(col_ha_dict)>0:
    col_ha = HeatmapAnnotation(**col_ha_dict,axis=1,
             verbose=0,#label_kws=dict(visible=False)
                               )
else:
    col_ha=None
    col_split=None
    col_split_order=None

cm1 = ClusterMapPlotter(
            data=tmp3c, top_annotation=col_ha, left_annotation=row_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,
            row_cluster_method='ward',row_cluster_metric='cosine',
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},xticklabels=True,
            col_split_gap=0.4,cmap='BlueDarkRed18',#turbo, parula, viridis,RedYellowBlue_r,BlueOrangeRed,
            # row_split=df_rows.Group,row_split_order=df_cols.MajorType.tolist(),
            row_split=row_split,row_split_gap=0.8,row_split_order=row_split_order,
            rasterized=True, #row_split_gap=0.1,vmax=1,vmin=0,center=0.5,
            plot=False,label='comp_score',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=1)


cm2 = ClusterMapPlotter(
            data=tmpcg, top_annotation=col_ha, #left_annotation=row_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,xticklabels=True,
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
            col_split_gap=0.4,row_split_gap=0.8,rasterized=True,
            plot=False,cmap='parula',label='mCG',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=0)

cm3 = ClusterMapPlotter(
            data=tmpch, top_annotation=col_ha, #left_annotation=row_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,xticklabels=True,
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
            col_split_gap=0.4,row_split_gap=0.8,rasterized=True,
            plot=False,cmap='agGrnYl',label='mCH',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=0) #or Greys,BlueOrange12,
if not atac_data is None:
    cm4 = ClusterMapPlotter(
            data=atac, top_annotation=col_ha, #left_annotation=row_ha,
            show_rownames=False, show_colnames=show_colnames,col_cluster=False,row_cluster=False,
            row_dendrogram=False, col_dendrogram=False,xticklabels=True,
            col_split=col_split, col_split_order=col_split_order,
            xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
            col_split_gap=0.4,row_split_gap=0.8,rasterized=True,
            plot=False,cmap='BlueDarkOrange12',label='ATAC\nlog(CPM)',
            standard_scale=0,legend_kws=dict(extend='both',extendfrac=0.1),
            verbose=0) #or Greys,BlueOrange12,
    cmlist=[cm1,cm4,cm2,cm3]
else:
    cmlist=[cm1,cm2,cm3]

In [ ]:
from PyComplexHeatmap.utils import despine
nrows,ncols=tmp3c.shape
width=max(ncols*0.5,5) if atac_data is None else max(ncols*0.8,5)
plt.figure(figsize=(width,12))
ax,legend_axes=composite(cmlist=cmlist, main=0,legend_hpad=2,col_gap=5,legend_gap=10)
cm1.ax.set_title("Boundary Prob",fontsize=14)
cm2.ax.set_title("mCG",fontsize=14)
cm3.ax.set_title("mCH",fontsize=14)
if not atac_data is None:
    cm4.ax.set_title("ATAC",fontsize=14)
ax.set_ylabel(str(tmp3c.shape[0])+" Diff Boundaries",color='black',fontsize=14)
ax.set_axis_on()
despine(ax=ax, left=True, bottom=True,top=True,right=True)
ax.tick_params(axis="both",which="both",left=False,
                right=False,labelleft=False,labelright=False,
                top=False,bottom=False,labeltop=False,labelbottom=False)
# ax.set_title("Title",y=1.03,fontdict={'fontweight':'bold','color':'red'})
plt.savefig(os.path.join(figdir,f'{group_col}_diff_tad_multiple_omics_heatmap.pdf'), transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
# # calculate pearson correlation for each cell type across all bins
# use_cols=[ct for ct in group_order if ct in mcg.columns.tolist()]
# mcg=mcg.loc[:,use_cols]
# mch=mch.loc[:,use_cols]
# atac=atac_data.loc[:,use_cols]
# rows=df_bin.index.tolist() # all bins
# print(len(rows))
# R=[]
# for ct in cols:
#     r_cg=pearsonr(bound_prob_ct.loc[rows,ct].values, mcg.loc[rows,ct].values)
#     r_ch=pearsonr(bound_prob_ct.loc[rows,ct].values, mch.loc[rows,ct].values)
#     r_atac=pearsonr(bound_prob_ct.loc[rows,ct].values, atac.loc[rows,ct].values)
#     R.append([ct,r_cg[0],r_cg[1],r_ch[0],r_ch[1],r_atac[0],r_atac[1]])
# data=pd.DataFrame(R,columns=['CellType','CG_corr','CG_pvalue','CH_corr','CH_pvalue','ATAC_corr','ATAC_pvalue']).set_index('CellType')
# data.to_csv(os.path.join(figdir,f'{group_col}.multiomics_correlation_across_bins.tsv'),sep='\t')

# df1=data.loc[:,['CG_corr','CH_corr','ATAC_corr']]
# df1.columns=[col.replace('_corr','') for col in df1.columns.tolist()]

# df_rows=pd.Series(group2parent)
# df_rows=df_rows.loc[df1.index.tolist()]
# colors={}
# for parent in df_rows.values:
#     if parent not in df_rows.tolist():
#         continue
#     colors[parent]=parent_colors[parent]
# row_ha_dict={
#     parent_col:anno_simple(df_rows,colors=colors,legend=False),
# }
# row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
#          verbose=0,label_side='bottom',label_kws=dict(visible=False))

# plt.figure(figsize=(3,5))
# cm = ClusterMapPlotter(
#                 data=df1, left_annotation=row_ha,
#                 show_rownames=True, show_colnames=True,col_cluster=False,row_cluster=True,
#                 row_dendrogram=False, col_dendrogram=False,
#                 row_split=df_rows,xticklabels=True,yticklabels=True,
#                 xticklabels_kws={'labelrotation':-30,'labelcolor':'black','labelsize':10,'bottom':True},
#                 yticklabels_kws=dict(right=True),
#                 row_split_gap=0.5,rasterized=True,#linecolor='white',linewidth=0.05,
#                 cmap='Spectral_r',label='Correlation',annot=True,
#                 legend_kws=dict(extend='both',extendfrac=0.1),# standard_scale=1,
#                 verbose=0)
# plt.savefig(os.path.join(figdir,f'{group_col}.multiomics_correlation_across_bins.pdf'), 
#             transparent=True,bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(2,2), dpi=300)
if 'ATAC_corr' in df_bin.columns.tolist():
    plot_data=[
        df_bin.loc[sig_bins, 'mCG_corr'],
        df_bin.loc[sig_bins, 'mCH_corr'],
        df_bin.loc[sig_bins, 'ATAC_corr'],
              ]
    xticklabels=['mCG', 'mCH', 'ATAC']
    colors=['limegreen','royalblue','darkorange']
else:
    plot_data=[
        df_bin.loc[sig_bins, 'mCG_corr'],
        df_bin.loc[sig_bins, 'mCH_corr']
    ]
    xticklabels=['mCG', 'mCH']
    colors=['limegreen','royalblue']
ax.set_yticks([-1,-0.5, 0,0.5,1])
ax.grid(axis='y',which='major',zorder=-1,linestyle='--')
sns.violinplot(plot_data, ax=ax,palette=colors)
ax.set_xticklabels(xticklabels)
# ax.set_title(group,fontsize=10)
ax.set_ylabel('Correlation with Boundary Prob',fontsize=8)
plt.savefig(os.path.join(figdir,f'{group_col}_multiomics_correlation_violinplot.pdf'), transparent=True,bbox_inches='tight')
plt.show()

In [ ]:
os.makedirs(os.path.join(figdir,"violinplot"), exist_ok=True)
data=df_bin.loc[sig_bins]
for group in data.Group.unique():
    df=data.loc[data.Group==group]
    fig, ax = plt.subplots(figsize=(2,2), dpi=300)
    if 'ATAC_corr' in df.columns.tolist():
        plot_data=[
            df['mCG_corr'],
            df['mCH_corr'],
            df['ATAC_corr'],
                  ]
        xticklabels=['mCG', 'mCH', 'ATAC']
        colors=['limegreen','royalblue','darkorange']
    else:
        plot_data=[
            df['mCG_corr'],
            df['mCH_corr']
        ]
        xticklabels=['mCG', 'mCH']
        colors=['limegreen','royalblue']
    ax.set_yticks([-1,-0.5, 0,0.5,1])
    ax.grid(axis='y',which='major',zorder=-1,linestyle='--')
    ax=sns.swarmplot(data=plot_data,palette=colors,
                      edgecolor='white',size=2)
    sns.violinplot(data=plot_data,ax=ax,
                   scale='count',linewidth=1,
                   inner=None,saturation=0.6,
                   color='white')
    ax.set_xticklabels(xticklabels)
    ax.set_title(group,fontsize=10)
    ax.set_ylabel('Correlation with Boundary Prob',fontsize=8)
    plt.savefig(os.path.join(figdir,"violinplot",f'{group}_multiomics_correlation_violinplot.pdf'), transparent=True,bbox_inches='tight',dpi=300)